# Pika H3 Flex 官方网页工作台
选择 A100 80GB / 高内存运行时。若已有测试正在生成，请先等它结束。
Colab Secrets 添加并授权 `CF_TUNNEL_TOKEN`；可选 `HF_TOKEN`。工作台无需密码。
运行下一格，通过你的独立工作台网页连接后端。默认无需密码；已有环境与模型缓存会复用。
Cloudflare Tunnel 服务地址应为 `http://127.0.0.1:8188`。首次模型下载发生在网页提交任务后。

In [ ]:
# -*- coding: utf-8 -*-
"""A100 官方 Pika 工作台一键启动：复用模型缓存，启动网页后端，无需密码。"""
from __future__ import annotations
import base64
import gzip
import json
import os
from pathlib import Path
import secrets
import shutil
import socket
import subprocess
import sys
import time
import urllib.request

PUBLIC_URL = "https://mcp-mac.id96.vip"
STUDIO_URL = "http://127.0.0.1:3091"  # 电脑上的本地独立网页；部署后可换为项目域名。
APP_VERSION = "1.3.0-a6ed78c"
PORT = 8188  # 复用原来 ComfyUI 的隧道服务端口。
STORAGE_ROOT = ""  # 空白自动检测 scratch；也可填 /mnt/local-scratch。
HTTP_DOWNLOAD = False  # Xet 下载异常时设 True，后续生成进程走普通 HTTP。
BUNDLE_GZ_B64 = "H4sIAAAAAAACCr19+XcTR9bov9Kf58tRCyRZsjGLE+UMAbO8EPDDTibzbD+dttSyO5bUSncL8Hh8jrOwhTVv2CEDTCBhMsOSTD5wWML/MuOW5Z/yL7y7VPWmliyYnElO4lZ3Lbeqbt2t7r210GfUbEerVHSrYDVqjlHVM/X5vmGl7zfKbr2mW5qjl5SyZVYVZ1ZXDumWUTbgDdSplbSKWdMVr4E3FVt3GnXFrFXmU0rNVLSGY1Y1xygqM9yUYdYyk7XJPvzXvXPGPfHYPfFI2WFWtGml9fJ86/bpX55fcx9caV76yT17c/XCTWXHvr2/PD+9eurvq387tXr2ofuXz+CnXYS2irOKe/+K++m9fy19yi1O1gjOQqHccBqWXigoRrVuWo6i1WqmQ73bkzXxblazZyvGtPf7I9useT9MW7RV1xwsJRsahZ9eIXu24RgV/2djum6ZRd32+7Dn/Wec2cnajgM7RwoHRz5Q8kpi03R5q64NZbdu3ZbTNg0O5LKb9S3Z8uYtQ6WBrVuKA1vK2YHB7OZSYrL2HlTbJ+tlS6Xy0JBW2pLdvKWo5fQtm7fmBgaGBqdLW6Y3bc5mS+WtW8pbhrYlsLv9u/buhkoLCdsxLW1GL1im6SSGlUQipSQczZ7DZ2fgkIa/Ld02Kw2cJ3y7aWu2jm9LDV45eDcEPzW7rhepidzm4W1YwNb1EvzeNADPxXqjYJbLFVPDV+NWQ4eXMC/VulMoGxVddl02LNspGFUASb6qaNE3ll7WLb0GcwpvJqbgzazj1Asl83BNdLBLq9j64mRt9zswyFxW2bBB2QYIVivpZQXQWdWsGTulbEgpxcOl/H7A1pSi1w6Jp4o5Q0/J4cmaAv9gYWimYtiOWtXqqu1YKXqZTHKBumXUHDXx3wibksh8ZBrcQzKllCsNezaP4xVlYU/Aqudp3TNVExDQrBlFVXw1yti7YtgKAiD6p2oeFmV8+BF4+I9hh//gzaxenAt2p8M8BJo5bDizilnXayp0g4uWwMpFs2TUZvKJhlNOb00kFc2GbVArVYI1eZxmEUAPwDJKbXWCxnZKZsPJB8vvHR2h97plBd+Pje888P54SnH0I06esSMKFryxLNOy87D89YpW1OHNdKNsG3/Q87lkGFDHmo+Ajv+UTQsWkSgTDSXD8MWU9BcVyyMspTxiXttqRv/hacsctgxHp7rdy1F7akwZC+eZYDysGU5cCcAUq9gBdEszbF05yGR7BGdNLSfcL1+sPLvj3vm+9eNdRT9iOPkFq7j4y/Pr7oPTzYs/Ni/fdV9eBjK7AJixmIj0qB8p6nVHeUez9RF6hH0/HAsUAV03KxU1GYPGUWzKOLpVNWrATNQOMxW/lKE2aIpwqIhsuaEODYkhBJBunKuMHKkbll5ar5c5A8fUCyhx64krEn5dhmFXKnGDC+BmplgxbW9uGCXLibUrj5sP/qe1dLl5+TGuWJSWKGkmM8OZXHnRjuItU0EklIWiWSsbM6okdPwTMK9kFB2VmYTsWgOCi3QLWZ1aIJpdKCQzxBoOAYQZUQJQYKZiTgP9hXczOpBFWRgIC1IjaiEDhEKOqljGLkX1fuBjuQKzeuDzzRPn146eWX3xIIN8OOERSaiTMWxq2IOeyWS9XkE5JE+MO4OjtFUsbekwXiQvaoSwpG1jJpEMrBk0DzMJyEsCDHAZVbaaoolJKkBHQKLxXtN8w2+evmRkRXkzfqBVGmIrJnhE7suja7efNa88dB/+1Pr+NogvK8sX3HPftT57sfLi5eqFe4rg0ivL91evfe7evwxrHtyX3F2mUS/h9vGA8Qt46EL9Nb+623x6HtEFpgP2/YnWN8dWr19qfX6teePva7cfr30me/zl+ck4WqcfQQZPc7vAosGwohKjTTG/BXaXMMR7puHe60q4OLWK3L4SKi5fA3+Pa37RX34a+gQLKlO8XDUPwMACtE8+VoEpPdO88LB5+pNEsq3NgLDjtbzgCT0gVNUTi1178BtQYDm5HwXrA3JTdb9PZ76uq7JjT56aIsrJXTuIbPg4pLyVV9qL4tvcUFd4ZGHFwzhl6J9Lf8oNKYBWSPkvPoqZBiHOBaZgIMdinRTvBocH8M+m4UH8MzS8iT4O5/jlEBehlwNcZNtwbjP/HcitM4fce+w6heaMRMzY+coG54uLKW8pAygJbh7s2jcWVlqnP3Ov/4j93/969fYD2T+KEHP6PE6IGpJp22TQIBmIQA0N+CBPm2ZlPYpRTixAnUV/Ad3lz9xHF9yl54F5UbElb2KCcvQU7KnQx4BEPZVMKv+V97bORHhnTXWdqLVvLrmPfnaXv21eOr7y7DFDt7J8VqE9BmTNfXohduKCcn9E5g/L+5FJDMwf6G4lRcUJJIYUnFmfMcST4l3wbb/p7DIbtVJofoeVhUA7ngwE1AjFf58+eMrHlDf5EYaBVVKkMBCzUCP0Kp/3aBzNPi0O1kl2ne+VZ0cVriYx4dz5lZ+vKz5Einvicuv2veCkw0ecdGw90PicUQNuZuv1FKmyMDwogCzcMZBUqInhRDtHhOI4GqzrUwVv2Q4ZJd3EB61RMszEIg3cWyDsZd2VCaG8e+7T1tJnjFrNy7eaF08g7wIwAwsD6nxNLIyUa2ZgU9YtnRQhTzQwamUzrLjQ9wJIWPWGo04kagC9oaXtqoEjSKc/bujWfBraysNg9SOpmlbVU1W9alrzGcd0tIr80bBBLChZxiHdKsB/NhJlagHmvqo5+aJ9KFUzZ0H40C14aNQMB/AmoOiEVcjdo+8r3KNCXSrUmfKe8U5Kwa74ifsbnpysJZSNNLgYdk27CgaNn0GatIw6CGQ2yAgO6iUgnU1kp/i3mkgF6AjSM5AasPbEANCpt5QtQ9lstg0zQ+pFwn3xJ/fkGQXhR6J55eHaXz5v3rgJHEbZmgUtvHn7a5BwWg+fADFtPbjDRh0QXzt0PAgdv61sym7bvE6/2KH75Af3zC2QmFqPj7ZeHlc27Tbe+dfSp9CZe/SE+8mN5v2/KDvMann+/b3Agt2jj1eeXWqevbd6/pj70+O14180L/yM+KzDNpxDnkijaN677f75FMDsHjuzdvxM68mXoCqBpMTyEoyxdfw794t7za+WWi+vr947BR3KsQBqSBmadEA5y8OAGblkSA2lvZHoR6zsh2q4VolkQFYNr5iUUrVDmlHRpis6La+jQs2JxHt6dbt8n5Ari4ucVDYouezAprAO4Z57CDO2+o9nq89uuseOuvev4PbyW+5Xdr9D+gP8jZMGYcH8wm8pOTSyYJV10IR65f5wDh//A2vufodxA5aLl0OskViC662XV5HpnX8IE+5evbfy8iv3wYmVF1fXjp/DBYJF/OrPrDI0byytvDjb+uaT5ve0HEwTbL0C3K0gTF0ROT0g84RsYUHeV5wFLawmtZ/44skMMFHgSrBNLTWgGLWRUW4MaWHJsNR4ShietWBHuL3Qvnnjnpiz239zj111H/x57erR5ulPWy9etJ48dH/+PBFvAAJdQQsrR52JImiopWrNYXpG+hc98Xd8Ht9+cPfIeGrX2PjvR0fCVC2op8CkGKijIBedmMJF8aCBxTkEfNNRDUev2tG5wJ2CH3CnUIEYZRn0XNAy5dJgKWSz+A6WJMYGhEBgKdZNyzYKZjiWRLyNpWz7rM6p1ss2WT71Q94zzh0ZO+dtfmF/3NDsWXheJBklUQaESMhGyjbM+mG9fdl7M3c0bEADZGRkYs6UDHuuQO9UHnIX8wSMhUoy+1LeziuDWd6yBKZpZ7QiIoFoKYVvflc48G6yCzjh1c1o9bpeK6kq91O2dORfDFd3m8yBMUL0bhYYDc3n0bc+6vBqFmeNSgkkIViGialgn1wQcR+xuqLDYiE6hTBE7k5vPNGN2ajBniiVaRPMjrdhV9zGbd74rvn1UvN/TrknHgHpg42K7OXtPEw98sQbf3dvPFq9foWZ1cqzs0qprLR+/pN7/CmzSUWeaPCeBkrH210JUgQgkEATVp5fExzpwk9AYleefskNhywGkoxVtSOqP9AUSub5iladLmkw04Y1TP9H1jGREzJuZ8zjRn2uAKKGfJchK5SNRmdgckA2Hb3m9JP0kkiuwylgQtaWTjZP/ZVnSQmf7chRHTatOZTNeWD9SmLUmNMKewYLuyr6kcKBctkoGlpF2I2KGhA4NF9kfaWJGoB6s+VEMo4kIwkqmxWU3pQCsEFEHtzIsDsOa5W5UP0Y6oVCHO17rBdn6mPhm2gXd5OE1rBWLDVCBMUqCKk9XwWxYO41KYmYi415bg8WioQNYJLGH/RfbbNa+scNtK0KlBsYIoKTUgaGBOlJC0AiBk5eZLJV0couKn9UVv/6dO3yD8qCT11CMgqU2LNLWX3+J6gLehwPrwchJtDcWx683XGznCBYfmQBBiUQEn5B4l19cJllj9Wn3ygL3uhDcPI4ggicqc4h2ulHQF8smKHjG6HiYClfmCnCWxVxRC79IdSbYI4BKXXQZCzg7TOiCHE2Kf6H1FWqFBhoO77QSeeMac5U9EyRzmLFmSWKOEhMw8UlFKr87AFBSnAbHAGc6nSmEMQkmgjqQ86EXrPxOPcQjFkFNHEK9fmUcohOnuh/FdMT8urzABh+wr06bYDeBH/r885s0Khcn8/QGtihPVU36gUSjMIKJLEDpHX1eTRZpqskDBh1lpGkKjgVPJvT6nT8zPKTOOWK6oESFNlphgdeNEs6mg2yUabE01KfF8IezQhU7gCrmCUfYK/Cvwm2Ufb7DoL8X2GQoRjgKKCF3jBKKtk+stJgJzjL4VmjCNwCOk2jDNeLeLz6/Kn76EtllJZTGczkBnip2XQoxOQHJ1tfH1V4ybFImoq458+ycpdos5snWrfvrr04H62CXgefP4Z67onr7rOngEcrLx+4d860rl5rnrrJrgidTgpZhJBDgxln2z3N/ccfh2Ydz6ARf7tUFv4VVHtetpGSaO0B3KVZatLbOQIlqE6K2Dg+JtepHqjZFY0iVX285b083QDRrVA0q1UQS9RixUhBmbqZEroZiKOEft5+JgtiwCoXNClO0QENwI8t4EbXj2jVOjBg3vVUkp5VYYXLOEecRMy5QsBOx0dXdOxBpaV0Wawic6PNBUDTxkpT7VSkMfpShT1RSSNYOFEoq6T3adN2Pz3tGUyj1MJTaOkos5LK5Tl50AcGPy1sp/xL1PDOLVKxhxlUyjs54BWOOXugYsL6n2o7DqCvZE4PNyAM8WEFEQuIdZtqU7aD1vOQrl0tSU0C2oJSaVkqbErGPabNkEFZjZqQ02l6kRZW5BSsdMCqTAXwt/ze2cwcoT4IG1A+hG0Ce08Fy061W13jbMaRocrmaAVRuYD/T0UMnNWS3CZ1wB0NiLFg81WQp9WAIQP0mky9ojlofETimgAJsXEksb6oHXW62of1yA7mnrnoHv2sees5m+wSQTcWEuziHVm6HCWHLbQhOT7eUONPqxSnYTnNGdK2Gb3oEVRyQYXSJP7RmUxYLJfiOgklXcQuZA0hcapo1udV/6M8dN2zq7DnwHsjeUT0oCaQUvDL++8UdmzfsaftM/1pTItiH46Mdy52BFmg19rOvWPb39k3glXyiVyIYoUPn6YEucrCHIzuHeX2ofrBUB/+TEEX4++NRr/jhMKX0d+P7zmw//3977y/a9fIwZGd2HVK2fH+zu2FD/aO7UWAdo58sHfHyFieOqQvQLTehw/7DmzfuXf/7nxi3/b/8/sEtXXg4I49BSqzfd++AzsKiBv5BNvO0JhYsPUZxGt7GBdE4ptjzpHmKqTfBMzI+IF3R/YHDwaxSGCxYZ0m/HJTiKpYwsPfal1iLwy6jA9q4o3fv1F9o5R+Y88b770xlkgqG4FUoI0dp4WK4v8KNZAQlTeUXJb+ERCg31ZekXNH+AkP5QRwyfQCdbeYgbeJiKbz5O7Kixur1x+4Ly6ivoP1FycnaxF3nBiBomrYtlHDPidoW4T3x4xBMkIddWY6YytX6/oMPs0UyV5V1eZgcr2jnaDwRbvDJ9iio3XEuJ7lNGFyfnLXPfoExoezi/Cw15zoK/kfF5o6zxdJJmkQowEn0ZbgS1jpkt5NuhKCFSItkGX9iF5sOGQtF4RaSEkFNPpPDA+QuKEOppTcgHCUCUvEvliXCCtyop/1LO9kH2ftNCgqe+or2qpO/gwsAA3unvGFTxiDojVasIKeuYFDHJK3/D1gmw2rqCdCwGKROC2L3jNBbls+sTZGzZBCBZZO9rLyomp6R7AieZhUTcYhrUSH96ZlzBg1eYxvD/f3Q1Ug0kD5q/3xgloGG++y/LpWilHDuoB0KA283Sao9oxs34ltr6N8SS+xqH5PfcOulG7FwQMAw3Lmu56FdgIQbUQNm6WnugkrW9GMWtxZpgCCepKg9UAemk/Pr976hElh88ZJtvs1L/zkfiGOOlZefrV68SqoXs2Tp/Ak6M7nq+eP+dZPcmdK9IwCZV1sdRQWK45u5acr5vRwzawJCbGk1x0qkAshiJzTfwf9aMbNhiM7cjQCZf2mD7FkIncY/kzX5weBJgTsHL3YRTxRZiIxun18D7FHqfJJGwmyP6DzaCDEo/+NJPmQoZ2qBA5NykCzdYt4GrQjXOgz9qw2MLRZVT3nduaj9XlsuCP+BWwqH0TOlDLkuqerWN9T7oSVjUQGVs34/HR63qEz08ysfqRkzOi255hZ1aw53QpMY0CaTmPd+aC3Ib2AwuSD5jNFaiPWB7HdkCabCJy6ifqBk94kHbjJaWTtM/CizWam+t4SKeVdfZ6ekh0NaOgvjGCETjVfyapFxV/PokVstWtPjmlW7H7Q+yrzhZJRxkMzyy5QPR1QkFl12gMBfc3ZPvDqwMjViLWv0WEYdRv9EmFj4ansYAoJShvpkmEj+0+j2C14vxgQfW7UZyytpAdqU9QMTQz+Ojyr65VexJ1/CwwrwYaX2J0VP/HdwFh3WV+9yc4jq5lIs5lD6WIcrzVf7Zgergo4PI2W7sk+GcQD7HD2TWk4HMdfw4kUvc0UpF9QoZBKoN7jfRHvM8VGSUu+qcBW1WVb9A6pi+djoSZl+4ECQIwLIIQaRdBJQGhXs0kodSTPJUz0GVEHB1KDA8kUl8onsFoiVcJTb1Fsmvxucpu99tUjvz0CkkWjqiYzeLaqYqOBTu35WnEWyKXxB/Ru6GuzBRbZLjWtd57AYsWIs8ejcDU7iCauRKBVMgbC8s7qlXpP+CJoKwVhMHElwltqVOu2uhCissNBEosrD7u94Nnehj2OvJiMiZMJGWlIaxYMFi2Bk7W+lNIHQ5mxgNxhDF0NYOIQOg5LOzAN631Ip/g5U5xUouFfJ+cgPDQFUqboJcNBNe+wbszMOjbIEXiy1cBjSPKOsBUWszMy1k2gZLlRK9Lui49k47CzydrowZFdez8kXJ4Y3fvu9sLowQO7D46MjU0p2Jh31FI1HLUOjB1oahXGo83Aw4YNdBxj+0ZZRB/R4kYlMOlkBKLq+XAj+fbGUlJ+0eyiYeSlC3kkhEGchTloJqKhwnoB2oUal3D91psLwAkNwJEVxApiS/ihrpfUDSIybMPcYYrgCtoT2ucgxFUIDWTTbQ2Fz/O4t8BA0EANy1wwGSmswKAC++11RiRCLFFK6DI49grJKxW9pvLXick+25ipanbh0GTfFMY45IKxBw0S9bKB0It1LILeFEK7YrSTsEUm+9znS+63p1g/XTt+vHnrCb4Hvate0aHBfFZ4Qebp/8mo/5BXsGA7el2tmrg5UopRQ7tg9NxA/gNSfsUswohpIOFvv1EO6rRHRMvoOq+VQUEgeyjJjEA0DHsWPSdQGKjqll6ZV3bAV2CuLDyEmwyTzLBJmWZyYz44u+y3BesaO50yugfxJ1SjfXLLk308o+jZDP0s9i/QJC6GZ5i+hWY5FXO67+GorRfNWsnOW+hBrQpAUwpwmW61QMmJrwmknydhg6JS52n6mURnydBqc9AczIiH0iCjzRiw7FahbFqAtaXCrGnOqWGUSHYXzcPbtstuCTiFxIZuiZg+NC34fn+dNnwVdGdPa5Arh6vqb4sTX+G2OH1JeDaSw+rqvVPu03OTfVLVD4gfoTfISguzgxmPnWi2x1qEykZhzKJY0bR0WRVXiJzAAZIUhYmh74uO1FljgnRIE23IFqWzYwFFGMcmVRIJc4fvODp+4pGii9TNu3iSunyq9eKFGC85f64++3zl2WNvvEHYMnsGd8ifIySmW0I/9PrvrTiCECzJQK0+v7R665PmufOrd562Hp5ZWT7LDumrP95qfnXeg4jnJ8PHl46l1Ww8eyHtkmHoVAD7wG/+en9xCwa/svyMRw4dri19unb0jHv1ntebvwoZ2ld6wX9TqGilEnfcxkx6qJfqIOJJw4Omwwzu1GnibBBq0RpXKHjD7PAdB1bSWW4KDrP17dcwu+jLu3yWn9du/mP1wjNvpNgeC3DV+qZQL95bbE3KZKLt8yeaJ84r741uUtyfHjNaRdoNoz2IoQLrcT/iJpGCWAFfcJXQK+8QiLdsCYRs7rv17bG1v3yJrsno2Xy2eev22nenMbT2BCML/gvKY4EEdZw4FLkK1GahMNk3LEVX7gJlR167kMiI5kdfVtyxb6/ycUNv6KS0wsLqabZSKVVz2gBKaTsYipGR5lvOzWCVlIoxbWnWPCVkyPwKORJmq1oxXsysAu9CXcN+tRQKlu7nSqADH7tjdgVjBmhxt2QLziwq64gjYdmXgEBjr0/4zDlDtwNfMrwEsgBGP+8ZHx89CHqxbjt7iN7DxhmXPeDHMaoiGmlYFTSEkVnXI834o/AxUEH4Sj8QNYKZHCb7ek3lgOu2fXS08MHIwbG9B/ZT3VxmMJNNa5v10patRSwwvn3s3TEM2pzsQ/cFwLW2sE1QZOWXcOAm0KhIFY7RDHC+yb5yJVxZhHFO9rHvRHuHiwD12OjIjnEEC+RMjCjkXYQhgvyUG87xA4Yd8tPg8AA/bBoe5Ieh4U3yzZAsI94MyDIYbQgC7GRt5MPxkf04TWIy6PAfgYPnTF3SkMxHdf9Jl4+H9en6ZN9im4Az2UdhT7IVoEqiAsgAftUqVkWmh9tRlj2syRLVugA1U67ARhIvN8HELcJM7d1J0Vl4EFFHU6M12fd/J7LpbVq6PLUwOLD430Re3tv+YeH9UTxuhdID5NmNMR8bNgww6SlWgOMqY0QQhn0lQdJp1dYr5ZSC7sSkvqIGD1qwfljoX8EQayiZ4YL0SKXpSVSR/qxYBo+PKWSCfGNEgTjvWjq1nOz7CFgWT4A42OcfjTpZTWGkEZlL9aDBs1Z2oBXHSBxQbstsEjGn/N5o0EoKcn2Jz/zxXLN2CIAhfXhs/P2dew8URrePjf3uwMGdDI/HTrwmbCA5yEwxxlGeUtsZOnAWlujBgWgd0EDQ/OoRqMzBfabvEuEVO6zN6aFiI4fI7SMKgWOiiNlDSZxk3AKLkfcaiMCHsCtM2hCdIk79EfOlrBkV4BjRFn+jNC/fWv3icXPpk+bJb1u3TwOz/+X5tZVnz9wvbmNc9I/P3BsiEmjt+Ol/gaxz/AyG+Jw/u3blpnvicnP56Opfn+Lx5JMfOBMAnjwib195dgaqCs3O7sc8P2GQeAJCM8E0WgQ35HleTTQTpZSSBhJ7LRYxuD67saueLsIKO8j4vGlgOoNoyfzUG7z74M+rZx/ByFsvr8I40Wf4/IvV61cw3p/Os1qPPsNArE9/al55uLL8dOXpMYUhw5PWlWcoG0EljJo68ci983nzwk+BxEYBgBgBCaaQ1QKzW8yL0x0yH0o3BjWJ5yhbgFJs3bzJc2MIKC2i6kYkR5N98Bf5fKamH1ajKJ8SZb0DmlTkEChyCuODDWKoAWpaAPgUu2l0P1Hh7lLI/qVfh4h4I1hRdWyrIJMX/GqjiNEjcYK5EYxfDEw1iWjUM9JxdOES7djo3CihS3Y/5dnuOJYx3XD02LMeAYI4p/KnmIknGt1nxQSTpc3vNZTsKD6gWdQAHEpKf929OzPlRqVSRXMHf183qBe0KtKemreOtx4ek0G9eLQLe7956dHK8hcrz2+FiCsdPsrkSyFqH+QKGUw5wkAwum6AlzGtTNSJ5dQpAxC9Q1/rjN0AkfoIHpijJAB7neSFwGygVYzKk69LrtdxRkL1ug5S4UgMeyIb7tpDXHEwNSuglXFkMrLMl24mZJWpXuFc/f4Za5xMnRHs0z+B6hkCUrrskpwe2cCUgIRQC/3+Q97z0lHXCwiY7ON3MZy0He+kf2sQ8XKY2YFWhD5KD4Mkvt4ajlTuMGpYC4yyOnpXyf1z6U9YR3HvX179+zeYEENq/CHIKKNBaBD4hofAQnXXYWBhfxDUmFg2Es57WCdigGgZeH7OS4oRWR47DKDv+itE5K3ZeqiGlxckVE2+xUrBVEoiI0eoLL8T8jrJ8F1nwUKDqZwEhFdGVkrgsB3MkAKCL80T5syQ8HTMieINgzKhyE+BbhlKv2cxEtG5UER6QhoZkHYS6Bam5DlxrPXzvdWzx/8FMs7lx2sXX8IWWr3wzP3paERAJL4TmDh8E50sKoVSMCehw62RUtK5KFEVjUkJ0wKyMK0DLVAHNmwIiZgy6QhW6JgghVqDv1B5aBBN/D2g4rdA1M5Tpo+n0Ejr+D+Ubdnslty2bQNDm7Zsym7blvMTy/zy/DqIOBjydO1c88bT0Jh/Q6HMn1wjw7l7Zrn14gWISUEbmru8vPbXZ80rP1N0+um1pZvu029Wnl9xj54AGQgxRYFJZ8tL69RdLshRWe6j79eunnGv/5ky74REROFOHnbp8DImpBT0DA8vV8C73KNaqWAB37k8nqpxbhZqRuZiwSqce4VIwAQShamesLB5+mTz5BLioiAHyE+8XCyhrpHZMVdExAqML4pVRrktHiwkCAdlCCELeAp0iAiV26iQ9HzHmZmYWodAhNOmMNXNBywJkXQpWAo5Af94W8kNrD+D7p1rIrUITd/ajSXiArkBQMfvmB+i0uFlICCC2/rmk7Wrn7vL38Qagb1zHDtiWcimghaCbMgGkA0oS8WKrtVoX09MhVcvPn1Lx+kLZChr/yiWBPO3EAL7pDjySdJGHtNwh3R2kWkNTAzLEyzfhSZJpp7hfDMTssOpKImLYJwPoMGQY7W4czR7Ar9MxRyniRmWkSULsuthkQqHmx4WYOHz1GIyEhVCHXirOwX4tg2nz/sgFho/DIY+iDX3PhC+kkBHEedtDW9sb7MH2bp5YwlwG2Byr/8M8vGgwoSRHtdu/gMeCa+/VxjV0c33ONDJL1eWl9znt6DS6snjzROX4Cd85bpxAiCf5ZPolQ9LYCmSbPIs7PjiR574vmTTefmQig8gZu6c95g2MKg8/i+lBAKG8uJvSgmQ5jzTt/hmfQqdx8dUIGlSXuBGWL9vTKOFP16oJW4dMHjJCEZhDov1Cuzg9Qobh7yvgZmKdAAXT6999Ze1v3ze+vYEe2KjrYMdrkUGlEByxOaNMyCwR5QJS6vaHExTznjiOY0hxHyfnXWf/ADCzNrFG+6dH9HT++kdYMDNHzD5oHv+NCOCYs9iioHTor9Lj2CR53TMhXPpkfvsqah88oz/3SiFLTJQv2BEBSB+GQJcuEOFC4qX60iWorkuCqoo0SZNecCFTXag56u5zTHSlIAnKlBJ2N9qk6D8UeUiaXg9G2CbbQMYIEd2eOY6mg05hjY+4NVAkiJ/TNCBstOwkaKwlE2HRSUWUdAhyTs4K+IsVthrYLE3et96eRWwQyz//a8BSRkDvEw+7rlLzccn3GNnGLPa+ABuokZV/ei14ST++BFW8aeJfYhQGXwblMHeRsLmRtwLz26LHFX3T7o/HwVayGNgrbhtANChjG8zgEZJFBQLnhd/gW5ZOiZLzwesQamOyRN4KvLBOaCjT3jDUKETCr7lbZ7nP8kYBorzMSGgQu9u+B1jWM5g4tJ2Ik+FfWrIptagCSne6PnwHMpQJ09QhiykHpy+DaiKNNa+JSB6O33oLTFDb+OxScCmGQElbPPxjgXQ3j/ZtwBwTCSMUmJqMX1oAa1v8IJ91GUQMtGDXHKRD2eCo6qYM12GFNs9n0+8et/QVbhvGekc23Wdlwt2Bi9vSELyLCq/DmTOESc46YKjB3wn6xO+xWYq4AU52UdukKFtwasjYQtiDY4wFPgqo6Y9puklAIbu6LxayqhqJOUwEk92d52M8a9Ewx+SCg7Dpp0yIa01U/zBj8JmqhIf3gPf2nYp1pbeoVg3EJqNnwKx2bxxLSEoCYjCJhmEK/hmKr6/oEGGWpwIGmmmRNO+KQa/y19iwGxu8Kvzb1mVl0l+D8drd4i3jlGJQyHXtHQxenEw7BoXql2lqceFXceHXnvYE1QVuH5QO01OJWNVKp5+XzvtFuvNi1fGQYT0FjI2D9PZSCw0AW0iUjMMVjjQ26MRxO18EhGRX7oKEcycfEbA1afa5pvJjM+ASduWvCdmGZCQyIzTklUJrhzmVpiUkbgn+SGyv2SIB0YPaCqdoPEkgFcDRwgNEh6WUTi9IGdsRC8dshi1cfbgXQfy8LNjsns2zPec9Z7PFb2s91ixQ3I0RIEyxtlhPHxd7ZaIrXuupw6J8ge6ZIv7N5LldxhkisfS7fKETqe1/iSED2v9c9m2vul4Nsxz4qWb3yLaGMWqDhyn5G+9eNjbfCHa551DoykOm5C3btrGkVjUhWJ4pUB9RmXkIfWFvJkyY3t3v7t33z7KP4gdi3Bg7+P4yMH34rYPFe50oUDc/QWR5JDr38wg8GKUkWKfac416nHpwDjuLEDK6BqD6Nl00G8hKoh2lFCl8i0O+YUe5uNOsWGhIFHgN2q7/dw/16eQc7kfBkP6Px/Bt0F8eBb96jxlX8KO6dh0J74vGgPtvFyy02e0QFjRxelK32mSgTcJPdFEd3BV/aibTkTkvjOxT4bSAH40TGWF7oIMq3NCOgajA0nArH9GraHH0m489OTaoWPPeK8UZg5Gu72wEyMIqI7CYT6kg/ms6vgx985xYVYhh2PPF7n51Wdrx8/EMohudIZzQYi1m4mTgDtTbZaNWSwWSkJ71XWTlEQTEcrc6928mnbsKoy/v38/SLSUMqPd2yrQMzC7ukpSITkPxEIHqxXNHUJYh6eFQZ2jZ4TvJrMEuX4X7sT6/94aUDerUQeUkFYA4ZjkSy3JLvy700VIsHavcetRd14az/Bj7knyFDLvpiQUcOmqJHonguVeozfaPYWaflg66uTVOCaX7HwhU8zdU6ClMFht+mTH+6fiLnLy1qP7bU6vcjlTLxc0BZ1DKMW3n+k0LpJuvSy66wtwgi+TthsOJcf+J/AULL7jqeT6Dfe083rdid3k9U7/CGuHGB9TCe/XK7ZCZD3cBLwVEXuxzKOrGNuBxL8ar4uRrXuK3+89F7LPeg5TEAhhpfAR6lya5qrNrzNmpQG3MEZmCnRd2HTU7no6gGh6/VCaLmpIhboWcTIyBvZ1YOCwE+V/H9Zrg+kP9uHZWOvbk62H91v3lhg+9+o9ye17BogCxesmHgLYs5pV+ndAE7YnxYsAck8cWz15XFFzm5QxajzZA2Rv/DEAAtr4QGJy+u22l3a/4bwWsCwlURga5/9ZWb6Pl0k9/Kb17Scya9XJHiClsKDXASEcL9QVa6nSOg2/Mu37teheG7Xix/gb/UK2iI7XzIUElNgr5P7T8tbr2Is6JYuv0J2HfpqOWCNzx8PXcOL4km6ub6buZTAyAsw7paFTMBn41Zu0ReAQ1JR/j36CMMWvKXogT8+RNN3dIW3jpwvwAqPX0c0hGLnmcUZ8HQZ/sdNK2PorrjlGCQQXvBwmJWtLS+7xp0BH+E7Kk3wO17x5d/XGKSYqnljLzvsOtKcGNMLkRHoz+n5OxV9c6SWYRtkS3g2/5m7wUNfTR7tsi/gm1tkR5Gri9UKGn/bZC22XmBqIRTDKqI2185WTPQ2eByxtuYxUAYW6CzlaR4YK6PmptnCTVAfhpM1AFLDeIXJ0MptHLON0LCZKhYxM8K7NqEVu2t3U+pidwVcG+DaauGM74cAuju4AlMX0oQ18aBhdQwEEtTqRzk2RkZC64JWPPxQUI2TneG63zZ9CXiwQ77rCllN5Ci20ZfJH+efSN0FcJ1SitkjXA744LXS6cnRmYOX0ORVvBsim4u4hUNLKwCbMsBlvxi1n2MKXIVECjwO7KMDJYBycCN1U4yM65dg5+rPge3DwUSEH0fXnMln/SPe3gLB13XLmfSTU6vU282HwTFmE90Kx6Jm0IMgCg8tVR+QiCLb0G2WnSauGpiYRuSrTQdvoYloyOEOMjG2zM20RDdQmek0yNwWaQXk/clPJsDo71K6/2o063XIUBDcAaHBEqKmKoZjTH6Wkh8NANpsSgLfFnwD0Xro0TiBDNWPywnihOm3heLVSAVhf3azZks7GluGb2WCQO/h2kvT4fF3wRcxaZRTpnLUfQXkTxH0M13Xy7QffXRrcp9dmnFl5vIrzjsNLJtepjXl509iGZVYYnpqZxsTI+nodf5gOjiV9oM7B26INu2aUy9Htz+sQZ70ONz2mO9A4luUEIvgUBcavYLfb85G2CAMMzUIQUbSGM4s5VotocW7bOt4dhMgq6nUvZjOeTqFpLEDyNYtSxdMuyYwZmAGBh6F2TQ4C9cjGwjRbDEo4pvnTQO7e8RTKgzYc5gbtonSGWQjEK2CmbKdPsrId/oY6Eo6BKHzEaV0IOHJwDFbckUyn8DBch/aTGaS+fLlc7NijqD3Zlw1PAipf2MZbQF0wGz8+vy0yJvfm5f79p837XzcvHW+9PO7e+ZYcU5+u/jUcQhBzmxnxP0IzYg4kJ3f1HcQ2hNt0r4DJm06fKv9r7MB+hS9wjvWVpUtS/LkumYXdI+Px52AydVBYlIEaowfGXq2KfBut047gQlpoxB7yhQxUNMU4Wk7xwcaqeGGaRSuk+qBxECNG1UKKpIspJRxsKnqTFrCeGpPmei/AjgLsN2U3tbcb9LZ9TXiHsoMxs/OOhZ6io0ZdThHsippOWYQO6sAl1snH2cPtN53Gzk6/7rGja5/dc59/6i4vB9Ul4SvMhaTuRKPIhtCElz2KIw0Ls4DJzBSqCI93ZiPRiimFbkjlohl+IxNbqPiKPgcrmbYTOdZihQmxOyg+olcFlvUU/DDpGdOL6V2YuTc9BkwEBUtWuyzTttM2vRruTIrjEGn5TPPsXffc5daTe6t/uyqdVQmdBiOX8pDojd31a3WjH2QfA4kz+X8AyG0pxg5304/4jh7h/Ibn1VqpRKbq6EGor5ohI+mgnmmOo6Pkh151Dp2PON7hCFaTQfo0i0adInGYFh5OO0CkN8cdv0oxUbRNPrW5bAflcJ1pdh991Xp4sfn3282Lj4iiXxM5nZeX3BPH1v7fTXETz8OLPPcD2+J8cvw8DdQPsa6kjCoVH2MisLrEl4pKYc/xuDhtWTIQHN4mi3jfkt2cf4LrMWHUUYv2lm+jMgFLMvVaU9x68HLt8gP34THOGMAXlwtMbnc9KEL3pNK8qYzRDzYkxGy5D9O7OLmaXkqPWqZjeruOUqVDPWGliFo84uA15xBYFM8WPfG/HJZ38gveNEmBKbn4JuURyfe/qeyBTg/UKvMAt1bVkQrkxxwL2PebynvakfT2GT2/Obtpaza7wINcjAq7kW0Mg62gEBNZMpRGJZ6FJdPoRua7pRcmKWGTURSGNJnMtFzRj6Q5ERNjphfRMKwEUvdQypMapWcT88PBYoGO8QO+6GBT5NIFeekclqZAuTYsTS62uSxhzRiUpaujha1K5oPxm+MXqJYBHcSsqVpp3v8cioPxYp3aane3j7JNyG9U2ojYv9XOi+Dd5Dp4h+Nox4GgkqsKZOAl6q/qJUPrj/dQ9YcRxoq4YIN1NizFSXAkenDzxuzZWO5jotduJ/bT6+aL7L1X2mfZdbaWMD/JxY+BsifmJvLVCAOaVzrk3NTdeQmdb3HfCffDiWFQRKZ6Wy8ewzABsdjraHtZD5/I+aFmgqPBaAZCy18V+dMtPRDSZAX67VeDKaGS/SobhP9ISZH72LgWBp7aaHcaphAWrSjyAlChzAyIinVbTcY5D6NRUUSLhhamY+iNlPPbba6SFonOfWt650ntPrHCf1p01L0nsol2w9RO+EEVh/3By4MJ7LI7vnjpS3pBGS6sRsXqjojBFCyMFV0QodvIAwZzb9uhZEAV0VoYhxfUYOwJCR88rTO7xDa8LqMndF5Udb+XcE1efcXGkug0iYRUpBJ5aaa83FPNH243b5xkjRLTMr04xmFLrZeoWq5+/cnq9Ssry1/Ap+bpkxzd5J57gscvN5aalx6tXvs8FPDY0eiPQ2kPZBK2fzmbZKJH6z8NLYq1FX1GK84HlyM2Mircmmgq4nwo703vdJKgir5IC8CnADunAjGOf4I9cts9xhFSoLH76FrzxneBdJldEUNeUf6KWNBRn0b97+s4W0I0iZKwY3PDgfGxKxy/F1Iz3eGMKQ9QLp5KhtQ6/Yh3YT3fRBdO6eMXFLHyNYzDUufYkRPzNjl6Dbm1HU78Q9n6haMvdmHUAkWTbZ6aYrGwj/UNX17SmdbDu7w7mreeu8/PhY3Rv4LVUKQmEdbDvOInNuwhtcOZi14Oh7UbS+6jcytPz2I2xPeMd3BjL1/AVA6fvYixJNIF0HHXm4c2WpIuig7BK26OJnA3KkNe3sXBXgL2KeCb6cuLq2vHz3k3SrNoKPIh//QYc+GdP9a8/C1PfQhyEW/USxCzt+fbqYfPjIB6+E1uRAwKB5djfW6II/EYd1Uvc1Z0wxU9sxge1ApvezU3kO1qeheniJYjTxEPy1PEeB9PS8fUucIbPvYKc3bd98p10NCnRVLGqB25atRUkVAz5TeSTHZ0mkE8nl7Hoyh2n2E2l5Xl+81L972sYWQV6dAVeviIAxXsLdnJdCCnJ825s+IK03SLA1NBY9dNek51GjW6iV5c3heTZTNOcOJ8Gx6qITWIpuRgGjqs8EXmeB/BH+g3/l2UMrJPoz2hISVslEWmOAW67YR9CSSHaI/leYVT8I6MhHZnN6O0oJAxZ92RixRQrS7Jw1K8aCFFddM5HHbg1gX0DipYWo1842KI7kH8FFXT/EqR8XUSJ9kHSZ2cLG1IpvkP0m+/nXg5gJsTRjWtNq+G9YnXUJmDbEgRY8NJDpE5shwAdOsrMARXXM4jh+mccE2gmUeuNp3sLYBK1sc6WmxwBFJrpClyUUXrtDoy3Au/tOdMoKbfpibommv6yUSvU/am8Gl4+2Stc4It5pmukyBMUDb0L9Ae7BwY0+EUOLzMEZukxPWB7OZf/Tg/SApw3rzE5JmZBsaPU3IPUo5AUmMX2eD5v1l0dCdtO8ALqq938A9F0rRYG3PrHP1vL+JxEM+6OLanWV+33/+oy4BcLVqu9T0HOuPSAs3KYnoByi/G4RVeJSmodu/97DRsjEqRgfPlhObgJYt4l9mbxCiQpwjjV3qB6DG+gZ4TPTo0dPZzio1dEQEl5PFEI27T7MoO3d3ooUlEICLxBUvF7HIptXhXkEQlFqyX7Hwi0k1QmYbW5jq53wX9OTqIHzSwiNQRewuKvHDAmhFZ8YV8YVM8g3yd2W7NNHAZR+mLXBEul9FKpYImCqgUtQ8CLiIA9KY1Kk6+Lb/37w4cfFdafWtOP90KlLaLFjIOygdR2DNYwHwQhQPi3gVfZenUKY4D2yS5gy4Wk91vzW3duk5lvLEzCPFkX25gSyYL/+a8bdGxY5FgvU/a8PJofgECUHDQIUZWRx8tkkKoFT63xXfepTWoFQgFFd8L3Ufc4qKGLnHzNAj/LlJSImB70+lof+AspG6Z8oxI3KyQj7s8QeVOcSJSBGsGJxSvQRf+ecEmUI2BZtgpTw0ksOeKPCHJ0KXb4h4NrkLpBnUSjc0a3fow3N+/4AGwOLzgQUC3JrXdwx3WXQRQ9AcvJUKzs3dniOfpMG1qVskLQgxmfvH8A9rFbWErl4OOs5bTAZ5XwMtOEHm3fqKCQAU/DDzSSjJ+0FbBizd4xStPFv8/rTdZiXSmAAA="


def health():
    try:
        with urllib.request.urlopen(f"http://127.0.0.1:{PORT}/api/health", timeout=3) as response:
            return json.load(response)
    except Exception:
        return {}


def live_owned(pidfile, expected):
    try:
        pid = int(pidfile.read_text().strip())
        cmdline = Path(f"/proc/{pid}/cmdline").read_bytes().replace(b"\0", b" ").decode(errors="replace")
        return pid if expected in cmdline else None
    except (OSError, ValueError):
        return None


def start_tunnel(work, get_secret):
    token = get_secret("CF_TUNNEL_TOKEN")
    if not token:
        print("⚠️ 服务已启动，但缺少 CF_TUNNEL_TOKEN。请在 Colab Secrets 添加并授权后重跑本格。", flush=True)
        return False
    pidfile = work / "studio-cloudflared.pid"
    if live_owned(pidfile, "cloudflared"):
        print("⏩ 本工作台隧道进程已在运行，连接状态见 studio-cloudflared.log", flush=True)
        return True
    binary = shutil.which("cloudflared")
    if not binary:
        local = Path("/content/cloudflared")
        if not local.is_file() or local.stat().st_size < 1024**2:
            partial = local.with_suffix(".part")
            machine = os.uname().machine
            arch = "arm64" if machine in {"aarch64", "arm64"} else "amd64"
            url = f"https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-{arch}"
            subprocess.run(["curl", "-fL", "--retry", "2", "--connect-timeout", "15", "--max-time", "120",
                            "-o", str(partial), url], check=True)
            partial.replace(local)
        local.chmod(0o755)
        binary = str(local)
    tunnel_env = os.environ.copy()
    tunnel_env["TUNNEL_TOKEN"] = token
    log = work / "logs" / "studio-cloudflared.log"
    with log.open("w", encoding="utf-8") as fp:
        # 使用环境变量传入令牌，避免在命令行与日志打印中暴露。
        proc = subprocess.Popen([binary, "tunnel", "--protocol", "http2", "run"],
                                stdout=fp, stderr=subprocess.STDOUT, env=tunnel_env, start_new_session=True)
    pidfile.write_text(str(proc.pid))
    start = time.monotonic()
    while time.monotonic()-start < 30:
        text = log.read_text(encoding="utf-8", errors="replace")
        if proc.poll() is not None:
            print("⚠️ 隧道退出，工作台仍在本地运行。日志：\n" + text[-8000:], flush=True)
            return False
        if "Registered tunnel connection" in text:
            print("✅ Cloudflare HTTP/2 隧道已注册连接", flush=True)
            return True
        time.sleep(.5)
    print(f"⚠️ 隧道连接尚未确认，请查看 {log}；本地工作台已就绪。", flush=True)
    return False


def main():
    if sys.platform != "linux":
        raise RuntimeError("本文件在 Colab A100 Linux 中运行；本地看界面请运行 preview.py")
    payload = json.loads(gzip.decompress(base64.b64decode(BUNDLE_GZ_B64)))
    namespace = {"__name__": "pika_installer", "__file__": "/content/pika-studio-installer.py"}
    exec(compile(payload.pop("installer_runtime.py"), "pika-studio-installer.py", "exec"), namespace)
    config = dict(namespace["CONFIG"], storage_root=STORAGE_ROOT, http_download=HTTP_DOWNLOAD)
    namespace["CONFIG"] = config
    current = health()
    if current.get("service") == "pika-h3-flex-studio":
        work = namespace["select_storage"](config)
        if current.get("version") == APP_VERSION:
            print("⏩ 已有同版本 Pika 工作台在线，无需密码，保留当前任务。", flush=True)
            start_tunnel(work, namespace["secret"])
            print(f"🌐 在电脑上打开工作台：{STUDIO_URL}\n后端隧道：{PUBLIC_URL}", flush=True)
            return
        # Only update our own idle server. Do not interrupt GPU work or queued jobs.
        for path in (work / "jobs").glob("*.json"):
            job = json.loads(path.read_text(encoding="utf-8"))
            if job.get("status") in {"running", "queued", "cancelling"}:
                raise RuntimeError("更新进度条前，请等当前队列完成或在网页取消任务，然后重跑这格；不会中断生成。")
        pid = live_owned(work / "studio-server.pid", str(work / "studio-app" / "server.py"))
        if not pid:
            raise RuntimeError("检测到旧版服务但无法确认归属，请先手动停止旧工作台服务器")
        import signal
        os.kill(pid, signal.SIGTERM)
        for _ in range(40):
            if not health():
                break
            time.sleep(.25)
        else:
            raise RuntimeError("旧版服务尚未停止，请稍后重跑")
        print("✅ 已停止本工作台的空闲旧版服务，正在更新。", flush=True)
    try:
        with socket.create_connection(("127.0.0.1", PORT), timeout=1):
            raise RuntimeError(f"{PORT} 已被其他程序占用，请先停止旧 ComfyUI，再运行工作台；本脚本不会杀其他进程。")
    except OSError:
        pass
    work, env, cli = namespace["prepare_environment"]()
    app_dir = work / "studio-app"
    for name, content in payload.items():
        target = app_dir / name
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_text(content, encoding="utf-8")
    env.pop("PIKA_STUDIO_PASSWORD", None)
    env.pop("CF_TUNNEL_TOKEN", None)
    log = work / "logs" / "studio-server.log"
    with log.open("a", encoding="utf-8") as fp:
        proc = subprocess.Popen([str(cli.parent / "python"), str(app_dir / "server.py"),
                                 "--root", str(work), "--port", str(PORT)],
                                stdout=fp, stderr=subprocess.STDOUT, env=env, start_new_session=True)
    (work / "studio-server.pid").write_text(str(proc.pid))
    for _ in range(120):
        if proc.poll() is not None:
            print(log.read_text(encoding="utf-8", errors="replace")[-10000:], flush=True)
            raise RuntimeError("工作台服务提前退出")
        if health().get("service") == "pika-h3-flex-studio":
            break
        time.sleep(.5)
    else:
        print(log.read_text(encoding="utf-8", errors="replace")[-10000:], flush=True)
        raise RuntimeError("工作台启动超时")
    print("✅ 工作台已启动，生成任务在网页提交，不会自动跑示例。", flush=True)
    print("🌐 工作台无需登录，打开网页即可使用。", flush=True)
    try:
        connected = start_tunnel(work, namespace["secret"])
    except Exception as exc:
        connected = False
        print(f"⚠️ 隧道准备失败：{exc}。本地工作台仍在运行。", flush=True)
    print(f"🌐 在电脑上打开工作台：{STUDIO_URL}\n后端隧道：{PUBLIC_URL}", flush=True)
    print(f"隧道 {'已连接' if connected else '尚待确认'}；Cloudflare 服务地址应为 http://127.0.0.1:{PORT}", flush=True)
    print(f"模型缓存复用：{work / 'hf'}\n服务日志：{log}\n生成视频：{work / 'outputs'}", flush=True)


if __name__ == "__main__":
    main()
